<a href="https://colab.research.google.com/github/AnuragRachoti/Foundation-Models-for-Computer-Vision/blob/main/Detr_Fine_tuning_by_Nikith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [16]:
!pip install torch torchvision transformers datasets matplotlib
!pip install -q torch torchvision pycocotools matplotlib opencv-python


Unzip the TUMTraf dataset

In [4]:
!unzip -q "/a9_dataset_r00_s01.zip" -d "/content/TUMTRAF_R1_S0"


 Inspect the Folder Contents

In [5]:
import os

for root, dirs, files in os.walk("/content/TUMTRAF_R1_S0"):
    print(f"📁 {root}")
    for file in files[:5]:  # only show first 5 files per directory
        print("   └──", file)


📁 /content/TUMTRAF_R1_S0
📁 /content/TUMTRAF_R1_S0/_calibration
   └── s40_camera_basler_north_16mm.json
   └── s50_camera_basler_south_50mm.json
   └── s50_camera_basler_south_16mm.json
   └── s40_camera_basler_north_50mm.json
📁 /content/TUMTRAF_R1_S0/_images
   └── 1616763340_947000000_s40_camera_basler_north_16mm.jpg
   └── 1616762540_963000000_s50_camera_basler_south_50mm.jpg
   └── 1616763301_112000000_s50_camera_basler_south_50mm.jpg
   └── 1616762591_258000000_s40_camera_basler_north_50mm.jpg
   └── 1616764491_100000000_s50_camera_basler_south_50mm.jpg
📁 /content/TUMTRAF_R1_S0/_labels
   └── 1611482141_223000000_s50_camera_basler_south_16mm.json
   └── 1616764451_148000000_s40_camera_basler_north_16mm.json
   └── 1616763870_779000000_s50_camera_basler_south_16mm.json
   └── 1616764341_049000000_s40_camera_basler_north_16mm.json
   └── 1611482320_970000000_s40_camera_basler_north_16mm.json


Path creation

In [10]:
image_dir = "/content/TUMTRAF_R1_S0/images"
annotation_file = "/content/TUMTRAF_R1_S0/_labels.json"


Custom Dataset Loader for TUMTraf R0 S1 Dataset

In [12]:
import os, json, torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class TUMTrafDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.samples = []

        for file in os.listdir(label_dir):
            if file.endswith('.json'):
                path = os.path.join(label_dir, file)
                with open(path, 'r') as f:
                    data = json.load(f)
                if 'labels' in data and data['labels']:
                    self.samples.append((data['image_file_name'], data['labels']))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, labels = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        boxes, class_ids = [], []
        for label in labels:
            box = label.get('box3d_projected')
            if not box: continue

            xs = [p[0] for p in box.values()]
            ys = [p[1] for p in box.values()]
            x_min, y_min = min(xs), min(ys)
            x_max, y_max = max(xs), max(ys)
            boxes.append([x_min * image.width, y_min * image.height, x_max * image.width, y_max * image.height])
            class_ids.append(0)  # CAR → class 0

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(class_ids, dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels}

        if self.transforms:
            image = self.transforms(image)

        return image, target


In [13]:
transform = T.Compose([
    T.Resize((480, 640)),
    T.ToTensor()
])

dataset = TUMTrafDataset(
    img_dir='/content/TUMTRAF_R1_S0/_images',
    label_dir='/content/TUMTRAF_R1_S0/_labels',
    transforms=transform
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))


Loading DETR and Fine-tuning

In [ ]:
!pip install -U torch torchvision --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201

In [ ]:
from torchvision.models.detection import detr_resnet50

model = detr_resnet50(pretrained=True)
model.class_labels = ["CAR"]  # Your dataset has only cars


In [17]:
import torchvision.models.detection
from torchvision.models.detection import detr_resnet50

model = detr_resnet50(pretrained=True)
model.class_labels = ["CAR"]  # Only one class
num_classes = 2  # CAR + background
in_features = model.class_embed.in_features
model.class_embed = torch.nn.Linear(in_features, num_classes)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.train()


ImportError: cannot import name 'detr_resnet50' from 'torchvision.models.detection' (/usr/local/lib/python3.11/dist-packages/torchvision/models/detection/__init__.py)